# 03 — Choisir le SFT : successif ou mixte

Je compare deux façons d'apprendre les mêmes tâches. **Je retiens le SFT mixte époque 3 pour conserver la QA tout en apprenant le triage**, avec un rappel urgent inférieur au successif sur validation.

Ce carnet décrit les entraînements réalisés ; ses cellules relisent uniquement leurs scores, sans relancer les calculs.
[02 · Évaluation](02_evaluation.ipynb) · [Parcours](README.md) · [04 · DPO](04_dpo.ipynb)


## 1. Le principe commun : apprendre un petit adaptateur

Je garde les poids de **Qwen3-1.7B-Base** figés et j'entraîne un LoRA : environ **17,4 millions de paramètres**, soit 1 % du modèle adapté, avec un rang de 16 et un alpha de 32. Le tokenizer transforme le texte en tokens ; la loss porte sur la réponse attendue et son token de fin, pas sur la consigne ni la demande.

```text
corpus du notebook 01 → encodage commun → entraînement LoRA
                     → checkpoints et adaptateur → inférence de validation
                     → métriques du notebook 02 → choix argumenté
```

| Opération | Code utilisé | Production |
|---|---|---|
| Préparer les tokens | [format_sft.py](../training/format_sft.py) | Prompt partagé, réponse complète, EOS natif |
| Réunir les tâches pour le mixte | [preparer_mixte.py](../training/preparer_mixte.py) | Train et validation mixtes, séparés ; test non lu |
| Entraîner | [entrainer_lora.py](../training/entrainer_lora.py) | Checkpoints, `adapter/`, protocole, résumé et suivi MLflow |
| Mesurer | [inference.py](../evaluation/inference.py), puis [evaluer.py](../evaluation/evaluer.py) | Prédictions JSONL et rapports JSON par checkpoint |

Les runs corrigés ont été lancés sur Colab, sauvegardés dans Drive puis récupérés localement. Les petits pilotes et les incidents ne changent pas cette chaîne.


## 2. Deux démarches, deux comportements

```text
A · SUCCESSIF : socle → LoRA QA → même LoRA spécialisé en triage
B · MIXTE    : socle → nouveau LoRA entraîné sur QA + triage ensemble
```

Dans A, la spécialisation triage dégrade fortement la QA : c'est un oubli de la tâche précédente. Dans B, chaque époque présente les deux tâches au même adaptateur, sans suréchantillonnage. Le mélange final contient 3 986 exemples d'entraînement ; la validation garde 500 exemples.

| Réglage | Successif, phase triage v3 | Mixte initial |
|---|---|---|
| Point de départ | LoRA QA corrigé | Nouveau LoRA depuis le socle |
| Train | 1 888 triages | 2 098 QA + 1 888 triages |
| Budget | 357 mises à jour | 3 époques, 750 mises à jour |
| Taux d'apprentissage | 0,0001 | 0,0001 |
| Batch effectif | 16 | 16 |
| Seed | 3407 | 3407 |

Ces recettes diffèrent aussi par leur exposition aux données et leur budget : je compare deux démarches, pas une ablation isolant uniquement l'ordre des exemples.


In [1]:
import json
from pathlib import Path

# Fonctionne depuis notebooks/ ou depuis la racine du dépôt.
RACINE = next(p for p in (Path.cwd(), *Path.cwd().parents)
              if (p / "serving/space/support/resultats.json").is_file())
def lire_json(relatif):
    return json.loads((RACINE / relatif).read_text(encoding="utf-8"))

resultats = lire_json("serving/space/support/resultats.json")
print("Source : bilans sauvegardés ; aucune inférence ni aucun entraînement relancé.")


Source : bilans sauvegardés ; aucune inférence ni aucun entraînement relancé.


## 3. Résultat qui motive le choix

La cellule lit le [bilan agrégé](../serving/space/support/resultats.json), dont chaque ligne indique les chemins et empreintes des rapports d'origine. Les deux candidats sont comparés sur les mêmes partitions de validation v3.


In [2]:
noms = {"SFT successif v3", "Mixte · époque 3"}
print(f"{'Modèle':<24} {'QCM':>9} {'Macro-F1':>10} {'Urgents':>10} {'ESI exact':>11}")
for ligne in resultats["lignes"]:
    if ligne["nom"] in noms:
        s = ligne["scores"]
        print(f"{ligne['nom']:<24} {s['accuracy_qcm']:>8.2%} {s['macro_f1_priorite']:>10.4f} {s['rappel_maximale']:>9.0%} {s['accuracy_esi']:>10.2%}")


Modèle                         QCM   Macro-F1    Urgents   ESI exact
SFT successif v3            3.82%     0.7715       95%     73.11%
Mixte · époque 3           57.25%     0.7874       91%     75.21%


**Décision.** Le mixte préserve les QCM (57,25 % contre 3,82 %) et obtient une macro-F1 légèrement supérieure. Il retrouve toutefois **91 urgences sur 100, contre 95** pour le successif. Je retiens ce compromis pour le POC bilingue, sans le présenter comme plus sûr cliniquement. Les neuf entrées urgentes manquées correspondent à sept cas distincts ; les traductions ne sont pas indépendantes.

Le test indépendant du carnet 02 concerne le successif uniquement. Une nouvelle confirmation indépendante du mixte reste nécessaire.


## 4. Essais intermédiaires : ce qu'ils ont appris

| Essai | Observation | Conséquence |
|---|---|---|
| Premier SFT QA local | Formats train/inférence différents, marqueur de fin mal adapté, réponses tronquées | Format commun, EOS natif, réponses complètes ; nouvel entraînement depuis le socle |
| Rappel QA pendant la spécialisation triage | QA récupérée, mais compromis triage insuffisant pour remplacer automatiquement la référence | Comparaison d'un mélange des tâches dès le départ |
| Mixte, époques 1 à 3 | Progression du triage ; époque 3 retenue comme compromis | Conservation du checkpoint 750 |
| Poursuite jusqu'à 10 époques | Aucun gain robuste simultané sur QA et urgences | Conservation de l'époque 3 |

Ces essais ne sont pas effacés : leurs détails sont dans les archives (archives locales non publiées). Ils sont résumés ici pour garder un seul fil de lecture.


## 5. Retrouver le modèle et passer au DPO

- Successif : `data/colab_archive/original/experiences/sft_dpo_v3_20261105/`, poids dans `sft_triage/adapter/`, évaluations dans `comparaison/sft_triage/`.
- Mixte : `data/colab_archive/sft_mixte_20261106_091447/`, rapports dans `validation/checkpoint-750/` ; poids sélectionnés dans le dépôt HF des adaptateurs, fournis avec la remise.
- Identité du modèle déployé : [modele.json](../serving/modele.json). Les archives volumineuses et MLflow ne sont pas inclus dans un clonage Git seul.

Je garde ces poids comme référence pour le DPO ciblé. Le carnet suivant examine si apprendre à préférer certaines réponses améliore réellement le triage.
